In [2]:
import tensorflow as tf
from keras.applications import EfficientNetV2B0, ResNet50, VGG16, Xception, MobileNetV3Small
from keras.optimizers import Adam
from keras.preprocessing import image_dataset_from_directory
from keras.losses import BinaryCrossentropy
import kagglehub
import os
from datasets import load_dataset

c:\Users\donof\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ds = load_dataset("ILSVRC/imagenet-1k")
pretrained_imagenet = load_dataset("timm/mini-imagenet")

In [4]:
LEARNING_RATE=0.0001
BATCH_SIZE=64
IMAGE_SIZE=160
VAL_SPLIT=0.2

In [5]:
efficientnetv2b0 = EfficientNetV2B0(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
resnet50 = ResNet50(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
vgg16 = VGG16(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
xception = Xception(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)
mobilenetv3small = MobileNetV3Small(weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False)

83683744/83683744 [==============================] - 2s 0us/step


4334752/4334752 [==============================] - 0s 0us/step


In [6]:
dataset_path = kagglehub.dataset_download("doctorstrange420/real-and-fake-ai-generated-art-images-dataset")
dataset_path

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1'

In [7]:
os.listdir(dataset_path+"\Data")

['FAKE', 'REAL']

In [8]:
fake_dir = os.path.join(dataset_path+"\Data", "FAKE")
fake_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1\\Data\\FAKE'

In [9]:
real_dir = os.path.join(dataset_path+"/Data", "REAL")
real_dir

'C:\\Users\\donof\\.cache\\kagglehub\\datasets\\doctorstrange420\\real-and-fake-ai-generated-art-images-dataset\\versions\\1/Data\\REAL'

In [10]:
os.listdir(real_dir)[:5]

['00060d29813e54eec710cd6f9948a40ac.jpg',
 '000698a1cfe903e219f81178d8e7795cc.jpg',
 '00071f2b6fc24b1885ddb1b53b512081c.jpg',
 '000ad71148c3ad9bb8f68d78a0aeac70a.jpg',
 '000ad71148c3ad9bb8f68d78a0aeac70b.jpg']

In [11]:
os.listdir(fake_dir)[:5]

['img000006.jpg',
 'img000013.jpg',
 'img000014.jpg',
 'img000015.jpg',
 'img000016.jpg']

In [12]:
trainingDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="training", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
trainingDs

Found 21642 files belonging to 2 classes.
Using 17314 files for training.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [13]:
valDs = image_dataset_from_directory(dataset_path+"\Data", seed=123, subset="validation", validation_split=VAL_SPLIT, batch_size=BATCH_SIZE, image_size=(IMAGE_SIZE, IMAGE_SIZE))
valDs

Found 21642 files belonging to 2 classes.
Using 4328 files for validation.


<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>

In [14]:
data_augmentation = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomZoom(0.1)
    ]
)
data_augmentation

In [15]:
vgg16.trainable=False

In [16]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.vgg16.preprocess_input(x)
x = vgg16.output
# x = vgg16(x, training=False)
x = tf.keras.layers.Flatten()(x)#GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [17]:
vgg16 = tf.keras.Model(inputs=vgg16.input, outputs=output)

In [18]:
vgg16.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_3 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 160, 160, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 160, 160, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 80, 80, 64)        0         
                                                                 
 block2_conv1 (Conv2D)       (None, 80, 80, 128)       73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 80, 80, 128)       147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 40, 40, 128)       0     

In [19]:
vgg16.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [20]:
efficientnetv2b0.trainable=False

In [21]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.efficientnet_v2.preprocess_input(x)
x = efficientnetv2b0.output
# x = efficientnetv2b0(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [22]:
efficientnetv2b0 = tf.keras.Model(inputs=efficientnetv2b0.input, outputs=output)

In [23]:
efficientnetv2b0.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 160, 160, 3)]        0         []                            
                                                                                                  
 rescaling (Rescaling)       (None, 160, 160, 3)          0         ['input_1[0][0]']             
                                                                                                  
 normalization (Normalizati  (None, 160, 160, 3)          0         ['rescaling[0][0]']           
 on)                                                                                              
                                                                                                  
 stem_conv (Conv2D)          (None, 80, 80, 32)           864       ['normalization[0][0]'] 

In [24]:
efficientnetv2b0.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [25]:
resnet50.trainable=False

In [26]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = resnet50.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [27]:
resnet50 = tf.keras.Model(inputs=resnet50.input, outputs=output)

In [28]:
resnet50.summary()

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 160, 160, 3)]        0         []                            
                                                                                                  
 conv1_pad (ZeroPadding2D)   (None, 166, 166, 3)          0         ['input_2[0][0]']             
                                                                                                  
 conv1_conv (Conv2D)         (None, 80, 80, 64)           9472      ['conv1_pad[0][0]']           
                                                                                                  
 conv1_bn (BatchNormalizati  (None, 80, 80, 64)           256       ['conv1_conv[0][0]']          
 on)                                                                                        

In [29]:
resnet50.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [30]:
xception.trainable = False

In [31]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = xception.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [32]:
xception = tf.keras.Model(inputs=xception.input, outputs=output)

In [33]:
xception.summary()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_4 (InputLayer)        [(None, 160, 160, 3)]        0         []                            
                                                                                                  
 block1_conv1 (Conv2D)       (None, 79, 79, 32)           864       ['input_4[0][0]']             
                                                                                                  
 block1_conv1_bn (BatchNorm  (None, 79, 79, 32)           128       ['block1_conv1[0][0]']        
 alization)                                                                                       
                                                                                                  
 block1_conv1_act (Activati  (None, 79, 79, 32)           0         ['block1_conv1_bn[0][0]'

In [34]:
xception.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [35]:
mobilenetv3small.trainable = False

In [36]:
# x = data_augmentation(tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
# x = tf.keras.applications.resnet50.preprocess_input(x)
x = mobilenetv3small.output
# x = resnet50(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

In [37]:
mobilenetv3small = tf.keras.Model(inputs=mobilenetv3small.input, outputs=output)

In [38]:
mobilenetv3small.summary()

Model: "model_4"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_5 (InputLayer)        [(None, 160, 160, 3)]        0         []                            
                                                                                                  
 rescaling_1 (Rescaling)     (None, 160, 160, 3)          0         ['input_5[0][0]']             
                                                                                                  
 Conv (Conv2D)               (None, 80, 80, 16)           432       ['rescaling_1[0][0]']         
                                                                                                  
 Conv/BatchNorm (BatchNorma  (None, 80, 80, 16)           64        ['Conv[0][0]']                
 lization)                                                                                  

In [39]:
mobilenetv3small.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss=BinaryCrossentropy(), 
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [40]:
history = efficientnetv2b0.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
)

Epoch 1/3



271/271 [==============================] - 157s 537ms/step - loss: 0.4354 - accuracy: 0.8019 - precision_1: 0.7961 - recall_1: 0.8090 - val_loss: 0.3095 - val_accuracy: 0.8833 - val_precision_1: 0.8869 - val_recall_1: 0.8841
Epoch 2/3
271/271 [==============================] - 144s 531ms/step - loss: 0.2905 - accuracy: 0.8839 - precision_1: 0.8836 - recall_1: 0.8829 - val_loss: 0.2444 - val_accuracy: 0.9069 - val_precision_1: 0.9119 - val_recall_1: 0.9049
Epoch 3/3
271/271 [==============================] - 137s 507ms/step - loss: 0.2438 - accuracy: 0.9049 - precision_1: 0.9022 - recall_1: 0.9071 - val_loss: 0.2112 - val_accuracy: 0.9214 - val_precision_1: 0.9188 - val_recall_1: 0.9280


In [41]:
history = vgg16.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=2
)

Epoch 1/2
 32/271 [==>...........................] - ETA: 6:20:12 - loss: 2.7844 - accuracy: 0.6660 - precision: 0.6686 - recall: 0.6712

KeyboardInterrupt: 

In [ ]:
history = resnet50.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
    
)

In [ ]:
history = mobilenetv3small.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=6
    
)

Epoch 1/3
271/271 [==============================] - 59s 196ms/step - loss: 0.5357 - accuracy: 0.7240 - precision_4: 0.7197 - recall_4: 0.7292 - val_loss: 0.4278 - val_accuracy: 0.8085 - val_precision_4: 0.8388 - val_recall_4: 0.7731
Epoch 2/3
271/271 [==============================] - 51s 187ms/step - loss: 0.4180 - accuracy: 0.8091 - precision_4: 0.8103 - recall_4: 0.8046 - val_loss: 0.3749 - val_accuracy: 0.8394 - val_precision_4: 0.8570 - val_recall_4: 0.8225
Epoch 3/3
271/271 [==============================] - 50s 185ms/step - loss: 0.3703 - accuracy: 0.8403 - precision_4: 0.8431 - recall_4: 0.8342 - val_loss: 0.3422 - val_accuracy: 0.8533 - val_precision_4: 0.8684 - val_recall_4: 0.8397


In [ ]:
history = xception.fit(
    trainingDs,
    validation_data=valDs,
    # training_steps=trainingDs.samples//trainingDs.batch_size,
    # validation_steps=valDs.samples//valDs.batch_size,
    epochs=3
    
)

Epoch 1/3
271/271 [==============================] - 431s 2s/step - loss: 1.6092 - accuracy: 0.5676 - precision_3: 0.5623 - recall_3: 0.5906 - val_loss: 0.6666 - val_accuracy: 0.6319 - val_precision_3: 0.6350 - val_recall_3: 0.6549
Epoch 2/3
  3/271 [..............................] - ETA: 5:45 - loss: 0.6944 - accuracy: 0.6458 - precision_3: 0.6604 - recall_3: 0.6863